In [ ]:
import warnings
warnings.filterwarnings(action="ignore")
import logging
from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [ ]:
import datetime as dt
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

# 通则和约定

QuantStudio 的对象通过 `args` 配置，因子数据按“时点 × 证券”组织，多因子数据使用 `Panel`。本篇用小数据介绍配置、复制和数据索引；计算图的使用见 [计算图框架](Core/计算图框架.ipynb)。

## 配置对象

显式 `args` 的优先级高于 JSON 配置文件，未提供的字段采用模型默认值。`qs_help()` 可查询对象的参数和方法，必要导入与业务示例都保留在导出文档中。

In [ ]:
from QuantStudio.Core import __QS_Object__
print(qs_help(__QS_Object__))

对象默认从用户目录下的 `QuantStudioConfig` 文件夹读取约定的配置文件，例如 `HDF5DBConfig.json`。可通过 `config_file` 指定文件，数据库凭据由使用者配置。

仅查询当前环境的默认配置位置：

In [ ]:
from QuantStudio import __QS_ConfigPath__
print(__QS_ConfigPath__)

对象常用属性包括 `Args`（参数集）、`Logger`（日志）和 `QSID`（语义身份）。影响结果的参数或依赖改变时，QSID 可能改变；`Name` 等排除字段只影响展示。QSID 不是运行实例编号，运行状态隔离与缓存复用见 [原理和细节](Core/原理和细节.ipynb)。

In [ ]:
print("QSID", qs_help(__QS_Object__.QSID), sep="\n")
print("-"*10)
print("Args", qs_help(__QS_Object__.Args), sep="\n")
print("-"*10)
print("Logger", qs_help(__QS_Object__.Logger), sep="\n")

In [ ]:
from QuantStudio.Factor.HDF5DB import HDF5DB

example_directory = TemporaryDirectory()
HDB = HDF5DB(args={"MainDir": example_directory.name})
print(HDB.Args.to_dict())
print(HDB.QSID)

## 复制与修改

`new(args=...)` 创建同类型的新对象，未覆盖的配置沿用原对象。对节点还可用 `deps` 替换依赖，详见 [计算图框架](Core/计算图框架.ipynb)。

In [ ]:
print(qs_help(__QS_Object__.new))

In [ ]:
NewFDB = HDB.new(args={"Name": "NewHDF5DB"})
print("QSID : ", NewFDB.QSID)
print("Args : ", NewFDB.Args)
print("Logger : ", NewFDB.Logger)

## 查询和更新参数

`Args` 是 Pydantic 参数模型，支持迭代、按名称读取和修改。只允许修改未冻结的字段；更新冻结字段时使用 `new(args=...)` 创建新对象。

- `QSID`：由参与身份计算的参数生成，不保证不同实例具有不同 QSID。
- `Logger`：记录参数校验与运行信息。
- `Owner`：参数所属对象，独立参数模型可以没有所属对象。

In [ ]:
# 参数获取和修改
print("修改前: ")
for iArgName, iArgVal in HDB.Args:
    print(iArgName, ": ", iArgVal)
print("=============================")

HDB.Args["FileOpenRetryNum"] = 3

print("修改后: ")
for iArgName, iArgVal in HDB.Args:
    print(iArgName, ": ", iArgVal)

参数集对象有如下基本方法：`to_dict`, `meta`, `info`

In [ ]:
from QuantStudio.Core import __QS_Args__

print(qs_help(__QS_Args__.to_dict))

In [ ]:
print(qs_help(__QS_Args__.meta))

In [ ]:
print(qs_help(__QS_Args__.info))

In [ ]:
print(HDB.Args.to_dict())
print(HDB.Args.meta(key="frozen"))
print(HDB.Args.info(html=False))

## Panel 数据索引

`Panel` 是 QuantStudio 的三维带标签数据容器。因子表数据的轴顺序为“因子 × 时点 × 证券”；风险协方差数据的轴顺序为“时点 × 证券 × 证券”。单个因子的 `DataFrame` 以日期为行、证券为列。

`iloc` 按位置索引，`loc` 按标签索引，`to_frame()` 转为多重索引 DataFrame，`values` 返回 NumPy 数组。

In [ ]:
# Panel 说明
from QuantStudio.Core.QSObject import Panel

print(qs_help(Panel))

In [ ]:
from QuantStudio.Core.QSObject import Panel

# 创建 Panel
DTs = pd.date_range(dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 4))
IDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
p = Panel(data=np.random.default_rng(0).random((2, 4, 3)), items=["close", "open"], major_axis=DTs, minor_axis=IDs)

print(p)

In [ ]:
# 通过位置索引数据
print(p.iloc[0, 0:3, [0, 2]])

In [ ]:
# 通过标签索引数据
print(p.loc["close", dt.datetime(2025, 1, 1):dt.datetime(2025, 1, 3), ["000001.SZ", "000003.SZ"]])

In [ ]:
# Panel 转 MultiIndex DataFrame
print(p.to_frame(filter_observations=False))

In [ ]:
# Panel 转 numpy.ndarray
print(p.values)